<a href="https://colab.research.google.com/github/aiza-snflwer/cardiometabolic-risk-prediction/blob/main/CAD.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
import pandas as pd
import numpy as np
import json

from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report, confusion_matrix, accuracy_score

from google.colab import files

In [2]:
uploaded = files.upload()
file_name = list(uploaded.keys())[0]

data = pd.read_excel(file_name)

print("Dataset shape:", data.shape)
data.head()

Saving CAD_clean.xlsx to CAD_clean.xlsx
Dataset shape: (303, 11)


,Age,Sex,BMI,HTN,Current Smoker,Typical Chest Pain,Q Wave,Fasting Blood Sugar,Bad Cholestrol,Protective Cholestrol,Outcome
0,53,0,29.387755,1,1,0,0,90,155,30.0,1
1,67,1,28.398718,1,0,1,0,80,121,36.0,1
2,54,0,20.077335,0,1,1,0,85,70,45.0,1
3,66,1,26.838648,1,0,0,0,78,55,27.0,0
4,50,1,37.165193,1,0,0,0,104,110,50.0,0


In [3]:
X = data[['Age',
          'Sex',
          'BMI',
          'HTN',
          'Current Smoker',
          'Typical Chest Pain',
          'Q Wave',
          'Fasting Blood Sugar',
          'Bad Cholestrol',
          'Protective Cholestrol']]

y = data['Outcome']

print(y.value_counts())

Outcome
1    216
0     87
Name: count, dtype: int64


In [4]:
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42
)

In [5]:
imputer = SimpleImputer(strategy="mean")

X_train = imputer.fit_transform(X_train)
X_test = imputer.transform(X_test)

In [6]:
scaler = StandardScaler()

X_train = scaler.fit_transform(X_train)
X_test = scaler.transform(X_test)

In [7]:
model = LogisticRegression(
    max_iter=1000,
    random_state=42
)

model.fit(X_train, y_train)

y_pred = model.predict(X_test)
y_prob = model.predict_proba(X_test)

risk_percentage = y_prob[:, 1] * 100

In [8]:
print("=== CAD MODEL ===")

print("Accuracy:", accuracy_score(y_test, y_pred))

print(classification_report(y_test, y_pred))

print(confusion_matrix(y_test, y_pred))

=== CAD MODEL ===
Accuracy: 0.8032786885245902
              precision    recall  f1-score   support

           0       0.71      0.56      0.62        18
           1       0.83      0.91      0.87        43

    accuracy                           0.80        61
   macro avg       0.77      0.73      0.75        61
weighted avg       0.80      0.80      0.80        61

[[10  8]
 [ 4 39]]


In [9]:
coefficients = model.coef_[0]

feature_names = [
    'Age',
    'Sex',
    'BMI',
    'HTN',
    'Current Smoker',
    'Typical Chest Pain',
    'Q Wave',
    'Fasting Blood Sugar',
    'Bad Cholestrol',
    'Protective Cholestrol'
]

feature_importance = pd.DataFrame({
    'Feature': feature_names,
    'Coefficient': coefficients
})

feature_importance = feature_importance.sort_values(
    by='Coefficient',
    ascending=False
)

print(feature_importance)

                 Feature  Coefficient
5     Typical Chest Pain     1.375501
0                    Age     1.101513
7    Fasting Blood Sugar     0.814186
6                 Q Wave     0.793514
3                    HTN     0.612777
4         Current Smoker     0.292797
8         Bad Cholestrol     0.164731
2                    BMI     0.078686
9  Protective Cholestrol     0.057354
1                    Sex    -0.359897


In [10]:
all_patients_json = []

for i in range(len(X_test)):

    patient = X_test[i]

    risk = risk_percentage[i]

    if risk > 50:
        level = "High"
    elif risk > 20:
        level = "Moderate"
    else:
        level = "Low"

    factors = []

    if patient[3] == 1:
        factors.append("Hypertension")

    if patient[4] == 1:
        factors.append("Smoking")

    if patient[5] == 1:
        factors.append("Typical Chest Pain")

    if patient[7] > 125:
        factors.append("High Blood Sugar")

    if patient[8] > 130:
        factors.append("High LDL Cholesterol")

    if patient[9] < 40:
        factors.append("Low HDL Cholesterol")

    # Example what-if simulation
    simulated_risk = max(risk - 15, 0)

    all_patients_json.append({
        "patient_id": i,
        "risk_percentage": round(float(risk), 2),
        "risk_level": level,
        "contributing_factors": factors,
        "simulated_lower_risk_if_lifestyle_improves":
            round(float(simulated_risk), 2)
    })

with open("CAD_PREDICTIONS.json", "w") as f:
    json.dump(all_patients_json, f, indent=2)

print("CAD JSON saved successfully.")

CAD JSON saved successfully.
